In [0]:
url = "jdbc:postgresql://ep-orange-voice-be1c0n8j-pooler.c-9.us-east-2.aws.neon.tech:5432/restaurante?sslmode=require"
usuario = "estudiante_01"
clave = "npg_fgeAi4HXC8Lq"

def consulta(sql):
    return (spark.read.format("jdbc")
        .option("url", url)
        .option("query", sql)
        .option("user", usuario)
        .option("password", clave)
        .option("driver", "org.postgresql.Driver")
        .load())

In [0]:
for capa in ["bronce", "plata", "oro"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{capa}")

In [0]:
for tabla in ["clientes", "productos", "sucursales", "pedidos", "detalle_pedido"]:
    consulta(f"SELECT * FROM public.{tabla}") \
        .write.mode("overwrite").saveAsTable(f"workspace.bronce.{tabla}")
    print("listo:", tabla)

listo: clientes
listo: productos
listo: sucursales
listo: pedidos
listo: detalle_pedido


In [0]:
for t in ["clientes", "productos", "sucursales", "pedidos", "detalle_pedido"]:
    print(t, spark.table(f"workspace.bronce.{t}").count())

clientes 34
productos 10
sucursales 2
pedidos 1020
detalle_pedido 1125


In [0]:
display(spark.sql("""
SELECT COUNT(*) AS filas, COUNT(DISTINCT pedido_id) AS pedidos_unicos
FROM workspace.bronce.pedidos
"""))

filas,pedidos_unicos
1020,1020


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.plata.pedidos AS
SELECT pedido_id, cliente_id, sucursal_id, fecha_pedido, canal,
  CASE
    WHEN LOWER(TRIM(estado)) IN ('completado','completada') THEN 'completado'
    WHEN LOWER(TRIM(estado)) = 'pendiente' THEN 'pendiente'
    WHEN LOWER(TRIM(estado)) = 'cancelado' THEN 'cancelado'
    WHEN LOWER(TRIM(estado)) = 'anulado'   THEN 'anulado'
    ELSE 'desconocido'
  END AS estado
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY pedido_id ORDER BY actualizado_en DESC) AS rn
  FROM workspace.bronce.pedidos
  WHERE pedido_id IS NOT NULL
)
WHERE rn = 1
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.plata.clientes AS
SELECT DISTINCT cliente_id, nombre, fecha_registro,
  COALESCE(NULLIF(INITCAP(TRIM(segmento)), ''), 'Sin segmento') AS segmento
FROM workspace.bronce.clientes
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.plata.detalle_pedido AS
SELECT detalle_id, pedido_id, producto_id, cantidad, precio_unitario,
       COALESCE(descuento, 0) AS descuento
FROM workspace.bronce.detalle_pedido
WHERE cantidad > 0 AND precio_unitario IS NOT NULL
""")

for t in ["productos", "sucursales"]:
    spark.sql(f"CREATE OR REPLACE TABLE workspace.plata.{t} AS SELECT DISTINCT * FROM workspace.bronce.{t}")

print("plata lista")

plata lista


In [0]:
# 1. Filas descartadas en el detalle
spark.sql("""
SELECT (SELECT COUNT(*) FROM workspace.bronce.detalle_pedido) AS bronce,
       (SELECT COUNT(*) FROM workspace.plata.detalle_pedido)  AS plata
""").show()

# 2. Estados normalizados
display(spark.sql("SELECT estado, COUNT(*) n FROM workspace.plata.pedidos GROUP BY 1 ORDER BY 2 DESC"))

# 3. Segmentos y categorías (¿hay repetidos con distinta escritura?)
display(spark.sql("SELECT segmento, COUNT(*) n FROM workspace.plata.clientes GROUP BY 1 ORDER BY 2 DESC"))
display(spark.sql("SELECT categoria, COUNT(*) n FROM workspace.plata.productos GROUP BY 1 ORDER BY 2 DESC"))

# 4. Rango del descuento
display(spark.sql("SELECT MIN(descuento) minimo, MAX(descuento) maximo FROM workspace.plata.detalle_pedido"))

+------+-----+
|bronce|plata|
+------+-----+
|  1125|  946|
+------+-----+



estado,n
completado,761
desconocido,142
pendiente,68
cancelado,43
anulado,6


segmento,n
Frecuente,12
Ocasional,11
Nuevo,9
Sin segmento,1
Frequente,1


categoria,n
Platos,3
Bebidas,2
,2
null,1
Postres,1
Sin categoría,1


minimo,maximo
-5.00,9999.00


In [0]:
# A. Valores de descuento que existen
display(spark.sql("SELECT descuento, COUNT(*) n FROM workspace.bronce.detalle_pedido GROUP BY 1 ORDER BY 1"))

# B. Por qué se descartaron 179 líneas
display(spark.sql("""
SELECT SUM(CASE WHEN cantidad IS NULL THEN 1 ELSE 0 END) AS cantidad_nula,
       SUM(CASE WHEN cantidad <= 0 THEN 1 ELSE 0 END) AS cantidad_cero_o_negativa,
       SUM(CASE WHEN precio_unitario IS NULL THEN 1 ELSE 0 END) AS precio_nulo,
       SUM(CASE WHEN precio_unitario <= 0 THEN 1 ELSE 0 END) AS precio_cero_o_negativo,
       MAX(cantidad) AS cantidad_max, MAX(precio_unitario) AS precio_max
FROM workspace.bronce.detalle_pedido
"""))

# C. Los 10 productos
display(spark.sql("SELECT * FROM workspace.bronce.productos ORDER BY producto_id"))

descuento,n
null,44
-5.00,43
-4.00,1
-2.00,1
-1.00,1
-0.25,34
0.00,485
0.05,60
0.10,126
0.15,60


cantidad_nula,cantidad_cero_o_negativa,precio_nulo,precio_cero_o_negativo,cantidad_max,precio_max
44,91,77,135,99999,999999.00


producto_id,nombre,categoria,precio_actual
1,Hamburguesa clásica,Platos,45.00
2,Ensalada,Platos,35.00
3,Limonada,Bebidas,15.00
4,Pastel,Postres,20.00
5,Pizza,Platos,65.00
6,Café,Bebidas,12.00
7,null,Sin categoría,0.00
8,,null,null
9,Prodcto mal escrto,,-12.50
10,N/A,,0.00


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.plata.clientes AS
SELECT DISTINCT cliente_id, nombre, fecha_registro,
  CASE
    WHEN LOWER(TRIM(segmento)) IN ('frecuente','frequente') THEN 'Frecuente'
    WHEN LOWER(TRIM(segmento)) = 'ocasional' THEN 'Ocasional'
    WHEN LOWER(TRIM(segmento)) = 'nuevo' THEN 'Nuevo'
    ELSE 'Sin segmento'
  END AS segmento
FROM workspace.bronce.clientes
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
# Valores altos de cantidad y precio (para fijar los límites de "atípico")
display(spark.sql("SELECT cantidad, COUNT(*) n FROM workspace.bronce.detalle_pedido WHERE cantidad > 10 GROUP BY 1 ORDER BY 1"))
display(spark.sql("SELECT precio_unitario, COUNT(*) n FROM workspace.bronce.detalle_pedido WHERE precio_unitario > 65 GROUP BY 1 ORDER BY 1"))

# Las 2 sucursales y cuánto detalle tiene cada producto
display(spark.sql("SELECT * FROM workspace.bronce.sucursales"))
display(spark.sql("SELECT producto_id, COUNT(*) n FROM workspace.bronce.detalle_pedido GROUP BY 1 ORDER BY 1"))

cantidad,n
99999,33


precio_unitario,n
65.65,2
66.30,4
66.95,2
67.60,2
68.25,4
68.90,2
69.55,1
70.20,4
999999.00,34


sucursal_id,nombre,ciudad
1,Centro,Ciudad Central
2,Norte,Ciudad Norte


producto_id,n
1,131
2,144
3,129
4,127
5,143
6,127
7,81
8,81
9,81
10,81


In [0]:
MAX_CANT = 100
MAX_PRECIO = 500

# Productos: los de categoría inválida quedan como "no identificado"
spark.sql("""
CREATE OR REPLACE TABLE workspace.plata.productos AS
SELECT producto_id,
  CASE WHEN categoria IS NULL OR TRIM(categoria) IN ('', 'Sin categoría')
       THEN 'Producto no identificado' ELSE TRIM(nombre) END AS nombre,
  CASE WHEN categoria IS NULL OR TRIM(categoria) = '' THEN 'Sin categoría'
       ELSE INITCAP(TRIM(categoria)) END AS categoria,
  CASE WHEN precio_actual > 0 THEN precio_actual END AS precio_actual
FROM workspace.bronce.productos
""")

# Detalle: líneas inválidas a cuarentena, con el motivo
spark.sql(f"""
CREATE OR REPLACE TABLE workspace.plata.detalle_cuarentena AS
SELECT *,
  CASE WHEN pedido_id IS NULL THEN 'pedido_id nulo'
       WHEN cantidad IS NULL THEN 'cantidad nula'
       WHEN cantidad <= 0 THEN 'cantidad cero o negativa'
       WHEN cantidad > {MAX_CANT} THEN 'cantidad atípica'
       WHEN precio_unitario IS NULL THEN 'precio nulo'
       WHEN precio_unitario <= 0 THEN 'precio cero o negativo'
       WHEN precio_unitario > {MAX_PRECIO} THEN 'precio atípico'
  END AS motivo
FROM workspace.bronce.detalle_pedido
WHERE pedido_id IS NULL OR cantidad IS NULL OR cantidad <= 0 OR cantidad > {MAX_CANT}
   OR precio_unitario IS NULL OR precio_unitario <= 0 OR precio_unitario > {MAX_PRECIO}
""")

# Detalle: líneas válidas, con el descuento corregido
spark.sql(f"""
CREATE OR REPLACE TABLE workspace.plata.detalle_pedido AS
SELECT detalle_id, pedido_id, producto_id, cantidad, precio_unitario,
  CASE WHEN descuento BETWEEN 0 AND 1 THEN descuento ELSE 0 END AS descuento,
  (descuento IS NOT NULL AND NOT (descuento BETWEEN 0 AND 1)) AS descuento_corregido
FROM workspace.bronce.detalle_pedido
WHERE pedido_id IS NOT NULL
  AND cantidad > 0 AND cantidad <= {MAX_CANT}
  AND precio_unitario > 0 AND precio_unitario <= {MAX_PRECIO}
""")

print("plata corregida")

plata corregida


In [0]:
display(spark.sql("SELECT motivo, COUNT(*) n FROM workspace.plata.detalle_cuarentena GROUP BY 1 ORDER BY 2 DESC"))
display(spark.sql("""
SELECT (SELECT COUNT(*) FROM workspace.bronce.detalle_pedido) AS bronce,
       (SELECT COUNT(*) FROM workspace.plata.detalle_pedido) AS plata,
       (SELECT COUNT(*) FROM workspace.plata.detalle_cuarentena) AS cuarentena
"""))

motivo,n
cantidad cero o negativa,91
precio cero o negativo,51
cantidad nula,44
precio atípico,34
cantidad atípica,33
precio nulo,28


bronce,plata,cuarentena
1125,844,281


In [0]:
display(spark.sql("""
SELECT MIN(fecha_pedido) AS fecha_min, MAX(fecha_pedido) AS fecha_max,
       SUM(CASE WHEN fecha_pedido IS NULL THEN 1 ELSE 0 END) AS fechas_nulas,
       SUM(CASE WHEN sucursal_id IS NULL OR sucursal_id NOT IN (1,2) THEN 1 ELSE 0 END) AS sucursal_invalida,
       SUM(CASE WHEN cliente_id IS NULL THEN 1 ELSE 0 END) AS cliente_nulo
FROM workspace.plata.pedidos
"""))
display(spark.sql("SELECT canal, COUNT(*) n FROM workspace.plata.pedidos GROUP BY 1 ORDER BY 2 DESC"))

fecha_min,fecha_max,fechas_nulas,sucursal_invalida,cliente_nulo
1900-01-01T00:00:00.000Z,2100-01-01T00:00:00.000Z,26,0,0


canal,n
Local,270
Delivery,257
Para llevar,248
,36
null,36
web,31
app??,25
??,25
SOCIAL,25
presencial,25


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.oro.fact_ventas AS
SELECT p.pedido_id,
       CAST(p.fecha_pedido AS DATE) AS fecha,
       CAST(DATE_TRUNC('month', p.fecha_pedido) AS DATE) AS mes,
       COALESCE(s.nombre, 'Sin sucursal') AS sucursal,
       COALESCE(s.ciudad, 'Sin ciudad') AS ciudad,
       COALESCE(c.segmento, 'Sin segmento') AS segmento,
       COALESCE(NULLIF(INITCAP(TRIM(p.canal)), ''), 'Sin canal') AS canal,
       COALESCE(pr.nombre, 'Producto no identificado') AS producto,
       COALESCE(pr.categoria, 'Sin categoría') AS categoria,
       (pr.nombre IS NOT NULL AND pr.nombre <> 'Producto no identificado') AS producto_identificado,
       d.cantidad,
       ROUND(d.cantidad * d.precio_unitario * (1 - d.descuento), 2) AS importe
FROM workspace.plata.pedidos p
JOIN workspace.plata.detalle_pedido d ON d.pedido_id = p.pedido_id
LEFT JOIN workspace.plata.productos pr ON pr.producto_id = d.producto_id
LEFT JOIN workspace.plata.sucursales s ON s.sucursal_id = p.sucursal_id
LEFT JOIN workspace.plata.clientes c ON c.cliente_id = p.cliente_id
WHERE p.estado = 'completado'
""")

# Pregunta 1: evolución mensual
spark.sql("""
CREATE OR REPLACE TABLE workspace.oro.ventas_mensuales AS
SELECT mes, COUNT(DISTINCT pedido_id) AS pedidos, ROUND(SUM(importe),2) AS importe_total
FROM workspace.oro.fact_ventas GROUP BY mes
""")

# Pregunta 2: productos y categorías por sucursal
spark.sql("""
CREATE OR REPLACE TABLE workspace.oro.productos_por_sucursal AS
SELECT sucursal, categoria, producto, producto_identificado,
       SUM(cantidad) AS unidades, ROUND(SUM(importe),2) AS importe_total
FROM workspace.oro.fact_ventas GROUP BY sucursal, categoria, producto, producto_identificado
""")

# Pregunta 3: segmentos
spark.sql("""
CREATE OR REPLACE TABLE workspace.oro.segmentos AS
SELECT segmento, COUNT(DISTINCT pedido_id) AS pedidos_completados,
       ROUND(SUM(importe),2) AS importe_total
FROM workspace.oro.fact_ventas GROUP BY segmento
""")

# Comparación de sucursales (objetivo general)
spark.sql("""
CREATE OR REPLACE TABLE workspace.oro.comparacion_sucursales AS
SELECT sucursal, COUNT(DISTINCT pedido_id) AS pedidos, SUM(cantidad) AS unidades,
       ROUND(SUM(importe),2) AS importe_total,
       ROUND(SUM(importe) / COUNT(DISTINCT pedido_id), 2) AS ticket_promedio
FROM workspace.oro.fact_ventas GROUP BY sucursal
""")

print("oro lista")

oro lista


In [0]:
display(spark.sql("""
SELECT (SELECT COUNT(*) FROM workspace.oro.fact_ventas) AS filas_fact,
       (SELECT COUNT(*) FROM workspace.plata.pedidos p
          JOIN workspace.plata.detalle_pedido d ON d.pedido_id = p.pedido_id
          WHERE p.estado = 'completado') AS filas_esperadas
"""))

filas_fact,filas_esperadas
756,756


In [0]:
display(spark.sql("SELECT * FROM workspace.oro.ventas_mensuales ORDER BY mes"))
display(spark.sql("SELECT * FROM workspace.oro.segmentos ORDER BY pedidos_completados DESC"))
display(spark.sql("SELECT * FROM workspace.oro.comparacion_sucursales"))
display(spark.sql("""
SELECT sucursal, categoria, ROUND(SUM(importe_total),2) AS importe
FROM workspace.oro.productos_por_sucursal GROUP BY 1,2 ORDER BY 1, 3 DESC
"""))

mes,pedidos,importe_total
null,1,20.00
2023-11-01,17,2813.30
2023-12-01,19,2843.20
2024-01-01,20,3185.15
2024-02-01,23,3258.40
2024-03-01,27,4509.15
2024-04-01,20,3045.20
2024-05-01,18,2793.85
2024-06-01,21,3465.50
2024-07-01,25,3615.85


segmento,pedidos_completados,importe_total
Frecuente,250,31848.71
Ocasional,218,28567.43
Nuevo,187,24820.39
Sin segmento,6,195.00


sucursal,pedidos,unidades,importe_total,ticket_promedio
Centro,331,1348,42518.48,128.45
Norte,330,1358,42913.05,130.04


sucursal,categoria,importe
Centro,Platos,33387.75
Centro,Bebidas,4956.49
Centro,Postres,3384.24
Centro,Sin categoría,606.00
Centro,Sin Categoría,184.00
Norte,Platos,33672.45
Norte,Bebidas,4743.73
Norte,Postres,4250.87
Norte,Sin categoría,246.00


In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.plata.productos AS
SELECT producto_id,
  CASE WHEN categoria IS NULL OR TRIM(categoria) IN ('', 'Sin categoría')
       THEN 'Producto no identificado' ELSE TRIM(nombre) END AS nombre,
  CASE WHEN categoria IS NULL OR TRIM(categoria) IN ('', 'Sin categoría')
       THEN 'Sin categoría' ELSE INITCAP(TRIM(categoria)) END AS categoria,
  CASE WHEN precio_actual > 0 THEN precio_actual END AS precio_actual
FROM workspace.bronce.productos
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.oro.fact_ventas AS
SELECT p.pedido_id,
       CAST(p.fecha_pedido AS DATE) AS fecha,
       CAST(DATE_TRUNC('month', p.fecha_pedido) AS DATE) AS mes,
       COALESCE(CAST(p.fecha_pedido AS DATE) BETWEEN DATE'2020-01-01' AND current_date(), FALSE) AS fecha_valida,
       COALESCE(s.nombre, 'Sin sucursal') AS sucursal,
       COALESCE(s.ciudad, 'Sin ciudad') AS ciudad,
       COALESCE(c.segmento, 'Sin segmento') AS segmento,
       COALESCE(pr.nombre, 'Producto no identificado') AS producto,
       COALESCE(pr.categoria, 'Sin categoría') AS categoria,
       (pr.nombre IS NOT NULL AND pr.nombre <> 'Producto no identificado') AS producto_identificado,
       d.cantidad,
       ROUND(d.cantidad * d.precio_unitario * (1 - d.descuento), 2) AS importe
FROM workspace.plata.pedidos p
JOIN workspace.plata.detalle_pedido d ON d.pedido_id = p.pedido_id
LEFT JOIN workspace.plata.productos pr ON pr.producto_id = d.producto_id
LEFT JOIN workspace.plata.sucursales s ON s.sucursal_id = p.sucursal_id
LEFT JOIN workspace.plata.clientes c ON c.cliente_id = p.cliente_id
WHERE p.estado = 'completado'
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.oro.ventas_mensuales AS
SELECT mes, COUNT(DISTINCT pedido_id) AS pedidos, ROUND(SUM(importe),2) AS importe_total
FROM workspace.oro.fact_ventas WHERE fecha_valida GROUP BY mes
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.oro.productos_por_sucursal AS
SELECT sucursal, categoria, producto, producto_identificado,
       SUM(cantidad) AS unidades, ROUND(SUM(importe),2) AS importe_total
FROM workspace.oro.fact_ventas
GROUP BY sucursal, categoria, producto, producto_identificado
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.oro.segmentos AS
SELECT COALESCE(c.segmento, 'Sin segmento') AS segmento,
       COUNT(DISTINCT p.pedido_id) AS pedidos_completados,
       ROUND(COALESCE(SUM(v.importe), 0), 2) AS importe_total
FROM workspace.plata.pedidos p
LEFT JOIN workspace.plata.clientes c ON c.cliente_id = p.cliente_id
LEFT JOIN (SELECT pedido_id, SUM(importe) AS importe
           FROM workspace.oro.fact_ventas GROUP BY pedido_id) v ON v.pedido_id = p.pedido_id
WHERE p.estado = 'completado'
GROUP BY 1
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.oro.comparacion_sucursales AS
SELECT sucursal, COUNT(DISTINCT pedido_id) AS pedidos, SUM(cantidad) AS unidades,
       ROUND(SUM(importe),2) AS importe_total,
       ROUND(SUM(importe) / COUNT(DISTINCT pedido_id), 2) AS ticket_promedio
FROM workspace.oro.fact_ventas GROUP BY sucursal
""")

print("oro lista")

oro lista


In [0]:
# Evolución mensual completa
display(spark.sql("SELECT * FROM workspace.oro.ventas_mensuales ORDER BY mes"))

# ¿Qué pasó en nov-dic 2024? (pedidos por estado alrededor de la caída)
display(spark.sql("""
SELECT CAST(DATE_TRUNC('month', fecha_pedido) AS DATE) AS mes, estado, COUNT(*) AS n
FROM workspace.plata.pedidos
WHERE fecha_pedido BETWEEN '2024-09-01' AND '2025-03-01'
GROUP BY 1, 2 ORDER BY 1, 2
"""))

# Top 5 productos por sucursal (pregunta 2)
display(spark.sql("""
SELECT sucursal, producto, categoria, unidades, importe_total FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY sucursal ORDER BY importe_total DESC) AS puesto
  FROM workspace.oro.productos_por_sucursal WHERE producto_identificado)
WHERE puesto <= 5 ORDER BY sucursal, puesto
"""))

mes,pedidos,importe_total
2023-11-01,17,2813.30
2023-12-01,19,2843.20
2024-01-01,20,3185.15
2024-02-01,23,3258.40
2024-03-01,27,4509.15
2024-04-01,20,3045.20
2024-05-01,18,2793.85
2024-06-01,21,3465.50
2024-07-01,25,3615.85
2024-08-01,27,4125.20


mes,estado,n
2024-09-01,cancelado,1
2024-09-01,completado,29
2024-09-01,desconocido,3
2024-09-01,pendiente,2
2024-10-01,cancelado,2
2024-10-01,completado,26
2024-10-01,desconocido,5
2024-10-01,pendiente,2
2024-11-01,completado,6
2024-11-01,desconocido,3


sucursal,producto,categoria,unidades,importe_total
Centro,Pizza,Platos,265,16467.10
Centro,Hamburguesa clásica,Platos,220,9465.30
Centro,Ensalada,Platos,226,7455.35
Centro,Pastel,Postres,184,3384.24
Centro,Limonada,Bebidas,211,2978.70
Norte,Pizza,Platos,256,15801.50
Norte,Ensalada,Platos,292,9711.10
Norte,Hamburguesa clásica,Platos,191,8159.85
Norte,Pastel,Postres,223,4250.87
Norte,Café,Bebidas,216,2470.48


In [0]:
display(spark.sql("SELECT * FROM workspace.oro.segmentos ORDER BY pedidos_completados DESC"))
display(spark.sql("SELECT * FROM workspace.oro.comparacion_sucursales"))
display(spark.sql("""
SELECT sucursal, categoria, ROUND(SUM(importe_total),2) AS importe
FROM workspace.oro.productos_por_sucursal GROUP BY 1,2 ORDER BY 1, 3 DESC
"""))
display(spark.sql("SELECT MAX(fecha) AS ultima_fecha_con_ventas FROM workspace.oro.fact_ventas WHERE fecha_valida"))

segmento,pedidos_completados,importe_total
Frecuente,297,31848.71
Ocasional,235,28567.43
Nuevo,187,24820.39
Sin segmento,42,195.00


sucursal,pedidos,unidades,importe_total,ticket_promedio
Centro,331,1348,42518.48,128.45
Norte,330,1358,42913.05,130.04


sucursal,categoria,importe
Centro,Platos,33387.75
Centro,Bebidas,4956.49
Centro,Postres,3384.24
Centro,Sin categoría,790.00
Norte,Platos,33672.45
Norte,Bebidas,4743.73
Norte,Postres,4250.87
Norte,Sin categoría,246.00


ultima_fecha_con_ventas
2026-10-31
